##CSV Data Analyst

In [0]:
# Option 1: Upload your CSV file
# Drag and drop your CSV file into the Databricks workspace, then update the path below
# Or use the default sample data

# Create a widget for file path input
dbutils.widgets.text("csv_file_path", "/Workspace/Users/abin.bijoy2@cognizant.com/Drafts/sales_data.csv", "CSV File Path")

# Get the file path from widget
csv_path = dbutils.widgets.get("csv_file_path")

print("═" * 70)
print("CSV DATA ANALYST - FILE UPLOAD OPTION")
print("═" * 70)
print("\nHow to use your own CSV file:")
print("\n  1. Drag and drop your CSV file into the workspace file browser")
print("  2. Right-click the file and select 'Copy Path'")
print("  3. Paste the path into the 'CSV File Path' widget above")
print("  4. Re-run this cell and the analysis cell below")
print("\n" + "═" * 70)
print(f"\nCurrently selected file: {csv_path}")
print("\nNote: Using default sample data unless you specify a different path.")
print("═" * 70)

In [0]:
import pandas as pd
import numpy as np

# Load the CSV and inspect its structure
df_raw = pd.read_csv(csv_path)

print("═" * 70)
print("CSV STRUCTURE DETECTED")
print("═" * 70)
print(f"\nFile: {csv_path}")
print(f"Rows: {len(df_raw):,}")
print(f"Columns: {len(df_raw.columns)}")
print("\n" + "═" * 70)
print("\nColumn Details:")
print("-" * 70)

for col in df_raw.columns:
    dtype = df_raw[col].dtype
    null_count = df_raw[col].isnull().sum()
    sample = df_raw[col].dropna().iloc[0] if not df_raw[col].dropna().empty else "N/A"
    print(f"  {col}")
    print(f"    Type: {dtype} | Nulls: {null_count} | Sample: {sample}")

print("\n" + "═" * 70)
print("\nFirst 5 rows:")
display(df_raw.head())

# Auto-detect date and amount columns
date_columns = []
amount_candidates = {}

for col in df_raw.columns:
    # Try to detect date columns
    if 'date' in col.lower() or 'time' in col.lower():
        date_columns.append(col)
    # Try to detect amount/sales columns with priority scoring
    if pd.api.types.is_numeric_dtype(df_raw[col]):
        col_lower = col.lower()
        priority = 0
        # High priority keywords (score 100)
        if any(keyword in col_lower for keyword in ['sales_amount', 'total_sales', 'revenue']):
            priority = 100
        # Medium priority keywords (score 50)
        elif any(keyword in col_lower for keyword in ['amount', 'sales', 'total']):
            priority = 50
        # Lower priority keywords (score 10)
        elif any(keyword in col_lower for keyword in ['price', 'value', 'cost']):
            priority = 10
        
        if priority > 0:
            amount_candidates[col] = priority

# Sort amount columns by priority (highest first)
amount_columns = sorted(amount_candidates.keys(), key=lambda x: amount_candidates[x], reverse=True)

print("\n" + "═" * 70)
print("AUTO-DETECTION RESULTS")
print("═" * 70)
print(f"\nDate columns found: {date_columns if date_columns else 'None - please specify'}")
print(f"Amount columns found: {amount_columns if amount_columns else 'None - please specify'}")
print("\n" + "═" * 70)

# Set default mappings or prompt for manual selection
if date_columns:
    detected_date_col = date_columns[0]
    print(f"\n✓ Will use '{detected_date_col}' as the date column")
else:
    detected_date_col = None
    print("\n⚠ No date column detected - you may need to specify manually")

if amount_columns:
    detected_amount_col = amount_columns[0]
    print(f"✓ Will use '{detected_amount_col}' as the sales amount column")
else:
    detected_amount_col = None
    print("⚠ No amount column detected - you may need to specify manually")

print("\n" + "═" * 70)

### Manual Column Override

If auto-detection doesn't work correctly, you can manually specify the columns in the analysis cell:

```python
# Manual override - uncomment and edit these lines in the analysis cell:
# detected_date_col = 'your_date_column_name'
# detected_amount_col = 'your_sales_column_name'
```

**Example scenarios:**
* CSV with columns like `transaction_date` and `total_revenue`
* CSV with `timestamp` and `gross_sales`
* CSV with unusual column names that auto-detection misses

In [0]:
# Use detected columns or allow manual override
date_col = detected_date_col if detected_date_col else 'order_date'  # fallback to default
amount_col = detected_amount_col if detected_amount_col else 'sales_amount'  # fallback to default

try:
    # Load and prepare data
    df = df_raw.copy()
    
    # Convert date column
    if date_col in df.columns:
        df[date_col] = pd.to_datetime(df[date_col])
    else:
        raise ValueError(f"Date column '{date_col}' not found in CSV")
    
    # Verify amount column exists
    if amount_col not in df.columns:
        raise ValueError(f"Amount column '{amount_col}' not found in CSV")
    
    # 1. Total Sales
    total_sales = df[amount_col].sum()
    
    # 2. Highest Revenue Month
    df['year_month'] = df[date_col].dt.to_period('M')
    monthly_revenue = df.groupby('year_month')[amount_col].sum().sort_values(ascending=False)
    highest_revenue_month = monthly_revenue.index[0]
    highest_revenue_amount = monthly_revenue.iloc[0]
    
    # 3. Average Order Value
    average_order_value = df[amount_col].mean()

    # Display results
    print("=" * 60)
    print("SALES DATA ANALYSIS RESULTS")
    print("=" * 60)
    print(f"\nUsing columns: Date='{date_col}', Amount='{amount_col}'")
    print("=" * 60)
    print(f"\n1. TOTAL SALES: ${total_sales:,.2f}")
    print(f"\n2. HIGHEST REVENUE MONTH: {highest_revenue_month}")
    print(f"   Revenue: ${highest_revenue_amount:,.2f}")
    print(f"\n3. AVERAGE ORDER VALUE: ${average_order_value:,.2f}")
    print("\n" + "=" * 60)
    
    # Show monthly revenue breakdown
    print("\nMonthly Revenue Breakdown (Top 5):")
    print("-" * 40)
    for month, revenue in monthly_revenue.head().items():
        print(f"{month}: ${revenue:,.2f}")
        
except Exception as e:
    print("═" * 70)
    print("⚠ ANALYSIS ERROR")
    print("═" * 70)
    print(f"\nError: {str(e)}")
    print("\nAvailable columns in your CSV:")
    print(list(df_raw.columns))
    print("\nPlease ensure your CSV has:")
    print("  - A date/time column")
    print("  - A numeric sales/amount column")
    print("\nOr manually specify the column names in the analysis code.")
    print("═" * 70)